# Notebook 01: Data Extraction And Market Context

## Project
NEM Network Constraints & Price Divergence Intelligence System

## Objective
Extract NSW1 and VIC1 dispatch price and region summary data from PostgreSQL, clean the data, align 5-minute intervals, and create the first market context table.

## Business Question
Do NSW1 and VIC1 have clean 5-minute dispatch data showing price, demand, available generation, and interchange for February 2026?


In [1]:
from pathlib import Path
import os

import pandas as pd
import numpy as np
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

pd.set_option("display.max_columns", 100)


In [2]:
PROJECT_ROOT = (
    Path.cwd().parents[0]
    if Path.cwd().name == "notebooks"
    else Path.cwd()
)

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "2026-02-01"
END_DATE = "2026-03-01"
REGIONS = ["NSW1", "VIC1"]

print("Project root:", PROJECT_ROOT)
print("Output directory:", OUTPUT_DIR)
print("Start date:", START_DATE)
print("End date:", END_DATE)
print("Regions:", REGIONS)


Project root: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence
Output directory: /Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs
Start date: 2026-02-01
End date: 2026-03-01
Regions: ['NSW1', 'VIC1']


In [3]:
load_dotenv(PROJECT_ROOT / ".env")

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

print("DB_USER:", DB_USER)
print("DB_HOST:", DB_HOST)
print("DB_PORT:", DB_PORT)
print("DB_NAME:", DB_NAME)


DB_USER: vivekarya
DB_HOST: localhost
DB_PORT: 5432
DB_NAME: nemweb


In [4]:
engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Database engine created.")


Database engine created.


In [5]:
with engine.connect() as conn:
    result = conn.execute(text("select current_database(), current_user"))
    for row in result:
        print(row)


('nemweb', 'vivekarya')


In [6]:
raw_tables_query = text("""
select
    table_schema,
    table_name
from information_schema.tables
where table_schema = 'raw'
order by table_name;
""")

raw_tables_df = pd.read_sql(raw_tables_query, engine)

raw_tables_df


,table_schema,table_name
0,raw,bid_day_offer
1,raw,bids_per_offer
2,raw,constraint_details
3,raw,constraint_equations
4,raw,constraint_rhs
5,raw,dispatch_constraints
6,raw,dispatch_load
7,raw,dispatch_price
8,raw,dispatch_regionsum
9,raw,dispatch_unit_scada


In [7]:
required_columns_query = text("""
select
    table_name,
    column_name,
    data_type
from information_schema.columns
where table_schema = 'raw'
  and table_name in (
      'dispatch_price',
      'dispatch_regionsum'
  )
order by table_name, ordinal_position;
""")

required_columns_df = pd.read_sql(required_columns_query, engine)

required_columns_df



,table_name,column_name,data_type
0,dispatch_price,settlementdate,timestamp without time zone
1,dispatch_price,regionid,text
2,dispatch_price,rrp,numeric
3,dispatch_price,intervention,text
4,dispatch_price,lastchanged,text
5,dispatch_price,raw_source_file,text
6,dispatch_price,raw_table_name,text
7,dispatch_price,eep,bigint
8,dispatch_price,rop,numeric
9,dispatch_price,raise6secrrp,numeric


In [8]:
START_DATE = "2026-02-01"
END_DATE = "2026-03-01"
REGIONS = ["NSW1", "VIC1"]

row_count_query = text("""
select
    'dispatch_price' as table_name,
    count(*) as row_count
from raw.dispatch_price
where settlementdate >= :start_date
  and settlementdate < :end_date
  and regionid = any(:regions)

union all

select
    'dispatch_regionsum' as table_name,
    count(*) as row_count
from raw.dispatch_regionsum
where settlementdate >= :start_date
  and settlementdate < :end_date
  and regionid = any(:regions);
""")

row_counts_df = pd.read_sql(
    row_count_query,
    engine,
    params={
        "start_date": START_DATE,
        "end_date": END_DATE,
        "regions": REGIONS,
    },
)

row_counts_df


,table_name,row_count
0,dispatch_price,16126
1,dispatch_regionsum,16126


In [9]:
region_check_query = text("""
select
    regionid,
    count(*) as row_count
from raw.dispatch_price
where settlementdate >= :start_date
  and settlementdate < :end_date
group by regionid
order by regionid;
""")

region_check_df = pd.read_sql(
    region_check_query,
    engine,
    params={
        "start_date": START_DATE,
        "end_date": END_DATE,
    },
)

region_check_df


,regionid,row_count
0,NSW1,8063
1,QLD1,8063
2,SA1,8063
3,TAS1,8063
4,VIC1,8063


In [10]:
dispatch_price_query = text("""
select
    settlementdate,
    regionid,
    rrp,
    intervention
from raw.dispatch_price
where settlementdate >= :start_date
  and settlementdate < :end_date
  and regionid = any(:regions)
order by regionid, settlementdate;
""")

dispatch_price_df = pd.read_sql(
    dispatch_price_query,
    engine,
    params={
        "start_date": START_DATE,
        "end_date": END_DATE,
        "regions": REGIONS,
    },
)

dispatch_price_df.head()


,settlementdate,regionid,rrp,intervention
0,2026-02-01 00:05:00,NSW1,57.05984,0
1,2026-02-01 00:10:00,NSW1,64.51979,0
2,2026-02-01 00:15:00,NSW1,65.08727,0
3,2026-02-01 00:20:00,NSW1,64.89000,0
4,2026-02-01 00:25:00,NSW1,63.47180,0


In [11]:
dispatch_price_df.shape


(16126, 4)

In [12]:
dispatch_regionsum_query = text("""
select
    settlementdate,
    regionid,
    totaldemand,
    availablegeneration,
    netinterchange,
    intervention
from raw.dispatch_regionsum
where settlementdate >= :start_date
  and settlementdate < :end_date
  and regionid = any(:regions)
order by regionid, settlementdate;
""")

dispatch_regionsum_df = pd.read_sql(
    dispatch_regionsum_query,
    engine,
    params={
        "start_date": START_DATE,
        "end_date": END_DATE,
        "regions": REGIONS,
    },
)

dispatch_regionsum_df.head()


,settlementdate,regionid,totaldemand,availablegeneration,netinterchange,intervention
0,2026-02-01 00:05:00,NSW1,8186.33,13272.53373,-1054.73,0
1,2026-02-01 00:10:00,NSW1,8165.29,13246.75420,-924.67,0
2,2026-02-01 00:15:00,NSW1,8202.70,13198.51267,-1006.50,0
3,2026-02-01 00:20:00,NSW1,8213.41,13169.71135,-1011.73,0
4,2026-02-01 00:25:00,NSW1,8055.53,13139.84760,-934.04,0


In [13]:
dispatch_regionsum_df.shape


(16126, 6)

In [14]:
print("dispatch_price_df:", dispatch_price_df.shape)
print("dispatch_regionsum_df:", dispatch_regionsum_df.shape)


dispatch_price_df: (16126, 4)
dispatch_regionsum_df: (16126, 6)


In [15]:
dispatch_regionsum_df["regionid"].value_counts()


regionid
NSW1    8063
VIC1    8063
Name: count, dtype: int64

In [16]:
dispatch_price_df["regionid"].value_counts()


regionid
NSW1    8063
VIC1    8063
Name: count, dtype: int64

In [17]:
def clean_dispatch_table(df):
    cleaned = df.copy()

    # Standardise column names
    cleaned.columns = cleaned.columns.str.lower()

    # Convert dispatch interval timestamp
    cleaned["settlementdate"] = pd.to_datetime(cleaned["settlementdate"])

    # Standardise region ID
    cleaned["regionid"] = (
        cleaned["regionid"]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    # Keep only non-intervention intervals
    if "intervention" in cleaned.columns:
        cleaned = cleaned[cleaned["intervention"].fillna(0).astype(int) == 0]

    # Remove duplicate region-interval rows
    cleaned = cleaned.drop_duplicates(
        subset=["settlementdate", "regionid"]
    )

    # Sort for time-series analysis
    cleaned = cleaned.sort_values(["regionid", "settlementdate"])

    cleaned = cleaned.reset_index(drop=True)

    return cleaned


In [18]:
dispatch_price_clean = clean_dispatch_table(dispatch_price_df)
dispatch_regionsum_clean = clean_dispatch_table(dispatch_regionsum_df)

print("Dispatch price before cleaning:", dispatch_price_df.shape)
print("Dispatch price after cleaning:", dispatch_price_clean.shape)

print("Dispatch region summary before cleaning:", dispatch_regionsum_df.shape)
print("Dispatch region summary after cleaning:", dispatch_regionsum_clean.shape)


Dispatch price before cleaning: (16126, 4)
Dispatch price after cleaning: (16126, 4)
Dispatch region summary before cleaning: (16126, 6)
Dispatch region summary after cleaning: (16126, 6)


In [19]:
dispatch_price_clean.head()


,settlementdate,regionid,rrp,intervention
0,2026-02-01 00:05:00,NSW1,57.05984,0
1,2026-02-01 00:10:00,NSW1,64.51979,0
2,2026-02-01 00:15:00,NSW1,65.08727,0
3,2026-02-01 00:20:00,NSW1,64.89000,0
4,2026-02-01 00:25:00,NSW1,63.47180,0


In [20]:
dispatch_regionsum_clean.head()


,settlementdate,regionid,totaldemand,availablegeneration,netinterchange,intervention
0,2026-02-01 00:05:00,NSW1,8186.33,13272.53373,-1054.73,0
1,2026-02-01 00:10:00,NSW1,8165.29,13246.75420,-924.67,0
2,2026-02-01 00:15:00,NSW1,8202.70,13198.51267,-1006.50,0
3,2026-02-01 00:20:00,NSW1,8213.41,13169.71135,-1011.73,0
4,2026-02-01 00:25:00,NSW1,8055.53,13139.84760,-934.04,0


In [21]:
dispatch_price_clean.isna().sum()


settlementdate    0
regionid          0
rrp               0
intervention      0
dtype: int64

In [22]:
dispatch_regionsum_clean.isna().sum()


settlementdate         0
regionid               0
totaldemand            0
availablegeneration    0
netinterchange         0
intervention           0
dtype: int64

## alidate 5-Minute Coverage

In [23]:
price_coverage = (
    dispatch_price_clean
    .groupby("regionid")
    .agg(
        first_interval=("settlementdate", "min"),
        last_interval=("settlementdate", "max"),
        intervals=("settlementdate", "count"),
        unique_intervals=("settlementdate", "nunique"),
    )
    .reset_index()
)

price_coverage


,regionid,first_interval,last_interval,intervals,unique_intervals
0,NSW1,2026-02-01 00:05:00,2026-02-28 23:55:00,8063,8063
1,VIC1,2026-02-01 00:05:00,2026-02-28 23:55:00,8063,8063


In [24]:
regionsum_coverage = (
    dispatch_regionsum_clean
    .groupby("regionid")
    .agg(
        first_interval=("settlementdate", "min"),
        last_interval=("settlementdate", "max"),
        intervals=("settlementdate", "count"),
        unique_intervals=("settlementdate", "nunique"),
    )
    .reset_index()
)

regionsum_coverage


,regionid,first_interval,last_interval,intervals,unique_intervals
0,NSW1,2026-02-01 00:05:00,2026-02-28 23:55:00,8063,8063
1,VIC1,2026-02-01 00:05:00,2026-02-28 23:55:00,8063,8063


In [25]:
market_context = dispatch_price_clean.merge(
    dispatch_regionsum_clean,
    on=["settlementdate", "regionid"],
    how="inner",
    suffixes=("", "_regionsum")
)

market_context.head()


,settlementdate,regionid,rrp,intervention,totaldemand,availablegeneration,netinterchange,intervention_regionsum
0,2026-02-01 00:05:00,NSW1,57.05984,0,8186.33,13272.53373,-1054.73,0
1,2026-02-01 00:10:00,NSW1,64.51979,0,8165.29,13246.75420,-924.67,0
2,2026-02-01 00:15:00,NSW1,65.08727,0,8202.70,13198.51267,-1006.50,0
3,2026-02-01 00:20:00,NSW1,64.89000,0,8213.41,13169.71135,-1011.73,0
4,2026-02-01 00:25:00,NSW1,63.47180,0,8055.53,13139.84760,-934.04,0


In [27]:
print("Dispatch price clean:", dispatch_price_clean.shape)
print("Dispatch region summary clean:", dispatch_regionsum_clean.shape)
print("Merged market context:", market_context.shape)


Dispatch price clean: (16126, 4)
Dispatch region summary clean: (16126, 6)
Merged market context: (16126, 8)


In [28]:
merge_coverage = (
    market_context
    .groupby("regionid")
    .agg(
        first_interval=("settlementdate", "min"),
        last_interval=("settlementdate", "max"),
        intervals=("settlementdate", "count"),
        unique_intervals=("settlementdate", "nunique")
    )
    .reset_index()
)

merge_coverage


,regionid,first_interval,last_interval,intervals,unique_intervals
0,NSW1,2026-02-01 00:05:00,2026-02-28 23:55:00,8063,8063
1,VIC1,2026-02-01 00:05:00,2026-02-28 23:55:00,8063,8063


In [29]:
market_context.isna().sum()


settlementdate            0
regionid                  0
rrp                       0
intervention              0
totaldemand               0
availablegeneration       0
netinterchange            0
intervention_regionsum    0
dtype: int64

In [30]:
market_context[
    [
        "settlementdate",
        "regionid",
        "rrp",
        "totaldemand",
        "availablegeneration",
        "netinterchange",
    ]
].head(10)


,settlementdate,regionid,rrp,totaldemand,availablegeneration,netinterchange
0,2026-02-01 00:05:00,NSW1,57.05984,8186.33,13272.53373,-1054.73
1,2026-02-01 00:10:00,NSW1,64.51979,8165.29,13246.75420,-924.67
2,2026-02-01 00:15:00,NSW1,65.08727,8202.70,13198.51267,-1006.50
3,2026-02-01 00:20:00,NSW1,64.89000,8213.41,13169.71135,-1011.73
4,2026-02-01 00:25:00,NSW1,63.47180,8055.53,13139.84760,-934.04
5,2026-02-01 00:30:00,NSW1,64.85825,8072.11,13075.02694,-1077.86
6,2026-02-01 00:35:00,NSW1,64.05037,7942.10,13037.29402,-1018.58
7,2026-02-01 00:40:00,NSW1,64.89000,7942.04,13036.20127,-1150.21
8,2026-02-01 00:45:00,NSW1,64.14536,7813.26,13047.50350,-1076.92
9,2026-02-01 00:50:00,NSW1,62.95677,7801.77,13065.23732,-1044.53


In [31]:
base_market_output = OUTPUT_DIR / "01_base_market_context.csv"

market_context.to_csv(base_market_output, index=False)

print("Saved base market context to:")
print(base_market_output)


Saved base market context to:
/Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/01_base_market_context.csv


In [32]:
base_market_summary = (
    market_context
    .groupby("regionid")
    .agg(
        intervals=("settlementdate", "count"),
        first_interval=("settlementdate", "min"),
        last_interval=("settlementdate", "max"),
        average_rrp=("rrp", "mean"),
        min_rrp=("rrp", "min"),
        max_rrp=("rrp", "max"),
        average_demand_mw=("totaldemand", "mean"),
        max_demand_mw=("totaldemand", "max"),
        average_available_generation_mw=("availablegeneration", "mean"),
        average_netinterchange_mw=("netinterchange", "mean"),
    )
    .reset_index()
)

base_market_summary


,regionid,intervals,first_interval,last_interval,average_rrp,min_rrp,max_rrp,average_demand_mw,max_demand_mw,average_available_generation_mw,average_netinterchange_mw
0,NSW1,8063,2026-02-01 00:05:00,2026-02-28 23:55:00,83.762187,-82.96215,20300.00000,7872.512299,12969.33,14385.235496,-434.561307
1,VIC1,8063,2026-02-01 00:05:00,2026-02-28 23:55:00,45.055455,-715.10671,269.77601,4692.430162,8786.93,10146.731392,654.223893


In [33]:
base_market_summary_output = OUTPUT_DIR / "01_base_market_context_summary.csv"

base_market_summary.to_csv(base_market_summary_output, index=False)

print("Saved base market summary to:")
print(base_market_summary_output)


Saved base market summary to:
/Users/vivekarya/Documents/GitHub/nem-network-constraints-price-divergence/outputs/01_base_market_context_summary.csv


## Analyst Note

Notebook 01 extracted NSW1 and VIC1 dispatch price and dispatch region summary data from PostgreSQL for February 2026.

The resulting base market context table contains one row per region per dispatch interval, combining RRP, total demand, available generation, and net interchange. This table has not yet added derived features or spike logic.

This clean merged dataset will be used as the input for Notebook 02, where base market features such as supply-demand gap, price spike flags, price changes, volatility, and time features will be created.
